In [ ]:
# NOTE: If you are running this notebook on Google Colab,
#       then uncomment the two lines below and then run this cell!

#!pip install datasets evaluate --upgrade
#!python -m spacy download de_core_news_sm

# 1 - Học Sequence to Sequence với Mạng Nơ-ron

Trong loạt bài này, chúng ta sẽ xây dựng một mô hình học máy để chuyển từ một chuỗi này sang một chuỗi khác, sử dụng PyTorch. Việc này sẽ được thực hiện trên bài toán dịch từ tiếng Đức sang tiếng Anh, nhưng các mô hình có thể được áp dụng cho bất kỳ bài toán nào liên quan đến việc chuyển từ một chuỗi sang chuỗi khác, chẳng hạn như tóm tắt văn bản, tức là chuyển từ một chuỗi thành một chuỗi ngắn hơn trong cùng một ngôn ngữ.

Trong notebook đầu tiên này, chúng ta sẽ bắt đầu đơn giản để hiểu các khái niệm cơ bản bằng cách triển khai mô hình từ bài báo [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215).

## Giới thiệu

Các mô hình sequence-to-sequence (seq2seq) phổ biến nhất là các mô hình _mã hóa-giải mã_ (encoder-decoder), thường sử dụng một _mạng nơ-ron hồi quy_ (RNN) để _mã hóa_ (encode) câu nguồn (đầu vào) thành một vector duy nhất. Trong notebook này, chúng ta sẽ gọi vector duy nhất này là _vector ngữ cảnh_. Chúng ta có thể hình dung vector ngữ cảnh là một biểu diễn trừu tượng của toàn bộ câu đầu vào. Vector này sau đó được _giải mã_ (decode) bởi một RNN thứ hai, RNN này học cách xuất ra câu đích (đầu ra) bằng cách tạo ra từng từ một.

![](assets/seq2seq1.png)

Hình trên minh họa một ví dụ về dịch máy. Câu đầu vào/nguồn, "guten morgen", được truyền qua lớp nhúng (màu vàng) và sau đó được đưa vào bộ mã hóa (màu xanh lá). Chúng ta cũng thêm token _bắt đầu chuỗi_ (`<sos>`) và _kết thúc chuỗi_ (`<eos>`) vào đầu và cuối câu tương ứng. Tại mỗi bước thời gian, đầu vào của RNN mã hóa bao gồm cả nhúng, $e$, của từ hiện tại, $e(x_t)$, và trạng thái ẩn từ bước thời gian trước đó, $h_{t-1}$, và RNN mã hóa xuất ra một trạng thái ẩn mới $h_t$. Chúng ta có thể hình dung trạng thái ẩn như một biểu diễn dạng vector của câu tính đến thời điểm hiện tại. RNN có thể được biểu diễn dưới dạng một hàm của cả $e(x_t)$ và $h_{t-1}$:

$$h_t = \text{EncoderRNN}(e(x_t), h_{t-1})$$

Chúng ta sử dụng thuật ngữ RNN một cách tổng quát ở đây, nó có thể là bất kỳ kiến trúc hồi quy nào, chẳng hạn như _LSTM_ (Long Short-Term Memory) hoặc _GRU_ (Gated Recurrent Unit).

Ở đây, chúng ta có $X = \{x_1, x_2, ..., x_T\}$, trong đó $x_1 = \text{<sos>}, x_2 = \text{guten}$, v.v. Trạng thái ẩn ban đầu, $h_0$, thường được khởi tạo bằng các giá trị 0 hoặc là một tham số được học.

Sau khi từ cuối cùng, $x_T$, được truyền vào RNN qua lớp nhúng, chúng ta sử dụng trạng thái ẩn cuối cùng, $h_T$, làm vector ngữ cảnh, tức là $h_T = z$. Đây là biểu diễn dạng vector của toàn bộ câu nguồn.

Bây giờ chúng ta đã có vector ngữ cảnh, $z$, chúng ta có thể bắt đầu giải mã nó để nhận được câu đích/đầu ra, "good morning". Một lần nữa, chúng ta thêm token bắt đầu và kết thúc chuỗi vào câu đích. Tại mỗi bước thời gian, đầu vào của RNN giải mã (màu xanh dương) là nhúng, $d$, của từ hiện tại, $d(y_t)$, cũng như trạng thái ẩn từ bước thời gian trước đó, $s_{t-1}$, trong đó trạng thái ẩn ban đầu của bộ giải mã, $s_0$, là vector ngữ cảnh, $s_0 = z = h_T$, tức là trạng thái ẩn ban đầu của bộ giải mã chính là trạng thái ẩn cuối cùng của bộ mã hóa. Do đó, tương tự như bộ mã hóa, chúng ta có thể biểu diễn bộ giải mã như sau:

$$s_t = \text{DecoderRNN}(d(y_t), s_{t-1})$$

Mặc dù lớp nhúng đầu vào/nguồn, $e$, và lớp nhúng đầu ra/đích, $d$, đều được hiển thị bằng màu vàng trong sơ đồ, chúng là hai lớp nhúng khác nhau với các tham số riêng biệt.

Trong bộ giải mã, chúng ta cần chuyển từ trạng thái ẩn thành một từ thực tế, do đó tại mỗi bước thời gian chúng ta sử dụng $s_t$ để dự đoán (bằng cách truyền qua một lớp `Linear`, được hiển thị bằng màu tím) từ mà chúng ta cho là từ tiếp theo trong chuỗi, $\hat{y}_t$.

$$\hat{y}_t = f(s_t)$$

Các từ trong bộ giải mã luôn được tạo ra liên tiếp, mỗi bước thời gian một từ. Chúng ta luôn sử dụng `<sos>` làm đầu vào đầu tiên cho bộ giải mã, $y_1$, nhưng đối với các đầu vào tiếp theo, $y_{t>1}$, đôi khi chúng ta sẽ sử dụng từ tiếp theo thực tế (ground truth) trong chuỗi, $y_t$, và đôi khi sử dụng từ được dự đoán bởi bộ giải mã, $\hat{y}_{t-1}$. Kỹ thuật này được gọi là _teacher forcing_ (ép buộc giáo viên), xem thêm thông tin về nó [tại đây](https://machinelearningmastery.com/teacher-forcing-for-recurrent-neural-networks/).

Khi huấn luyện/kiểm tra mô hình, chúng ta luôn biết có bao nhiêu từ trong câu đích, vì vậy chúng ta dừng tạo từ khi đạt đến số lượng đó. Trong quá trình suy luận (inference), thông thường việc tạo từ sẽ tiếp tục cho đến khi mô hình xuất ra token `<eos>` hoặc sau khi một số lượng từ nhất định đã được tạo ra.

Sau khi có câu đích được dự đoán, $\hat{Y} = \{ \hat{y}_1, \hat{y}_2, ..., \hat{y}_T \}$, chúng ta so sánh nó với câu đích thực tế, $Y = \{ y_1, y_2, ..., y_T \}$, để tính toán hàm mất mát. Sau đó, chúng ta sử dụng mất mát này để cập nhật tất cả các tham số trong mô hình.


## Chuẩn bị Dữ liệu

Đầu tiên, nhập tất cả các thư viện cần thiết. Các thư viện chính mà chúng ta sẽ sử dụng là:

-   [PyTorch](https://pytorch.org/) để tạo các mô hình
-   [spaCy](https://spacy.io/) để hỗ trợ phân tách từ (tokenization) dữ liệu
-   [torchtext](https://github.com/pytorch/text) để cung cấp một số hàm hỗ trợ
-   [datasets](https://huggingface.co/docs/datasets/index) để tải và xử lý dữ liệu
-   [evaluate](https://huggingface.co/docs/evaluate/index) để tính toán các chỉ số đánh giá


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

Chúng ta sẽ đặt tất cả các seed ngẫu nhiên có thể để có kết quả xác định.


In [ ]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

### Tập dữ liệu

Tiếp theo, chúng ta sẽ tải tập dữ liệu bằng thư viện `datasets`. Khi sử dụng hàm `load_dataset`, chúng ta truyền tên của tập dữ liệu, `bentrevett/multi30k`.

Tập dữ liệu chúng ta sử dụng là một tập con của [tập dữ liệu Multi30k](https://github.com/multi30k/dataset), được lưu trữ [tại đây](https://huggingface.co/datasets/bentrevett/multi30k) trên HuggingFace dataset hub. Tập con này có khoảng 30.000 câu song song tiếng Anh và tiếng Đức được lấy từ dữ liệu thô của nhiệm vụ 1 [tại đây](https://github.com/multi30k/dataset/tree/master/data/task1/raw). Chúng ta sử dụng phiên bản "2016" của các tập kiểm tra.


In [ ]:
dataset = datasets.load_dataset("bentrevett/multi30k")

Chúng ta có thể thấy đối tượng `dataset` (một `DatasetDict`) chứa các phần train, validation và test, số lượng ví dụ trong mỗi phần, và các đặc trưng trong mỗi phần ("en" và "de").


In [ ]:
dataset

Để thuận tiện, chúng ta tạo một biến cho mỗi phần. Mỗi biến là một đối tượng `Dataset`.


In [ ]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

Chúng ta có thể truy cập vào từng `Dataset` để xem một ví dụ cụ thể. Mỗi ví dụ có hai đặc trưng: "en" và "de", là các câu song song tiếng Anh và tiếng Đức.


In [ ]:
train_data[0]

### Bộ phân tách từ (Tokenizers)

Tiếp theo, chúng ta sẽ tải các mô hình spaCy có chứa bộ phân tách từ.

Bộ phân tách từ được sử dụng để chuyển một chuỗi thành một danh sách các token tạo nên chuỗi đó, ví dụ: "good morning!" trở thành ["good", "morning", "!"]. Từ bây giờ, chúng ta sẽ nói về các câu là một chuỗi các token thay vì nói chúng là một chuỗi các từ. Sự khác biệt là gì? "good" và "morning" đều là từ và token, nhưng "!" không phải là từ. Chúng ta có thể nói "!" là dấu câu, nhưng thuật ngữ token tổng quát hơn và bao gồm: từ, dấu câu, số và bất kỳ ký hiệu đặc biệt nào.

spaCy có mô hình cho từng ngôn ngữ ("de_core_news_sm" cho tiếng Đức và "en_core_web_sm" cho tiếng Anh) cần được tải để chúng ta có thể truy cập bộ phân tách từ của mỗi mô hình.

**Lưu ý**: các mô hình phải được tải trước bằng cách sử dụng các lệnh sau trên dòng lệnh:

```
python -m spacy download en_core_web_sm
python -m spacy download de_core_news_sm
```

Chúng ta tải các mô hình như sau:


In [ ]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

Chúng ta có thể gọi bộ phân tách từ cho mỗi mô hình spaCy bằng phương thức `.tokenizer`, phương thức này nhận một chuỗi và trả về một chuỗi các đối tượng `Token`. Chúng ta có thể lấy chuỗi từ đối tượng token bằng thuộc tính `text`.


In [ ]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

Tiếp theo, chúng ta sẽ viết một hàm để áp dụng bộ phân tách từ cho tất cả các ví dụ trong mỗi phần dữ liệu, cũng như áp dụng một số xử lý khác.

Hàm này nhận vào một ví dụ từ đối tượng `Dataset`, áp dụng bộ phân tách từ tiếng Anh và tiếng Đức của spaCy, cắt giảm danh sách token xuống độ dài tối đa, tùy chọn chuyển mỗi token thành chữ thường, và sau đó thêm token bắt đầu chuỗi và kết thúc chuỗi vào đầu và cuối danh sách token.

Hàm này sẽ được sử dụng với phương thức `map` từ một `Dataset`, phương thức này cần trả về một từ điển chứa tên các đặc trưng trong mỗi ví dụ nơi các đầu ra được lưu trữ. Vì tên đặc trưng đầu ra "en_tokens" và "de_tokens" chưa có trong ví dụ (nơi chúng ta chỉ có các đặc trưng "en" và "de"), điều này sẽ tạo ra hai đặc trưng mới trong mỗi ví dụ.


In [ ]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

Chúng ta áp dụng hàm `tokenize_example` bằng phương thức `map` như dưới đây.

Đối số `example` được ngầm hiểu, tuy nhiên tất cả các đối số bổ sung cho hàm `tokenize_example` cần được lưu trong một từ điển và truyền vào đối số `fn_kwargs` của `map`.

Ở đây, chúng ta cắt giảm tất cả các chuỗi xuống độ dài tối đa 1000 token, chuyển mỗi token thành chữ thường, và sử dụng `<sos>` và `<eos>` làm token bắt đầu và kết thúc chuỗi tương ứng.


In [ ]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Bây giờ chúng ta có thể xem một ví dụ, xác nhận hai đặc trưng mới đã được thêm vào; cả hai đều là danh sách chuỗi chữ thường với token bắt đầu/kết thúc chuỗi được thêm vào.


In [ ]:
train_data[0]

### Từ điển (Vocabularies)

Tiếp theo, chúng ta sẽ xây dựng _từ điển_ cho ngôn ngữ nguồn và ngôn ngữ đích. Từ điển được sử dụng để liên kết mỗi token duy nhất trong tập dữ liệu của chúng ta với một chỉ số (một số nguyên), ví dụ: "hello" = 1, "world" = 2, "bye" = 3, "hates" = 4, v.v. Khi đưa dữ liệu văn bản vào mô hình, chúng ta chuyển chuỗi thành các token và sau đó chuyển token thành các số sử dụng từ điển như một bảng tra cứu, ví dụ: "hello world" trở thành `["hello", "world"]` và sau đó trở thành `[1, 2]` sử dụng các chỉ số ví dụ trên. Chúng ta làm điều này vì mạng nơ-ron không thể hoạt động trên chuỗi, chỉ hoạt động trên các giá trị số.

Chúng ta tạo từ điển (một từ điển cho mỗi ngôn ngữ) từ các tập dữ liệu bằng hàm `build_vocab_from_iterator`, được cung cấp bởi `torchtext`, hàm này nhận một iterator trong đó mỗi phần tử là một danh sách các token. Sau đó, nó đếm số lượng token duy nhất và gán cho mỗi token một giá trị số.

Về lý thuyết, từ điển của chúng ta có thể đủ lớn để có một chỉ số cho mỗi token duy nhất trong tập dữ liệu. Tuy nhiên, điều gì sẽ xảy ra nếu một token tồn tại trong tập validation và test nhưng không có trong tập huấn luyện? Trong trường hợp đó, chúng ta thay thế token bằng một "token không xác định", được ký hiệu là `<unk>`, token này có chỉ số riêng (thường là chỉ số 0). Tất cả các token không xác định đều được thay thế bằng `<unk>`, ngay cả khi các token khác nhau, tức là nếu các token "gilgamesh" và "enkidu" đều không có trong từ điển, thì chuỗi "gilgamesh hates enkidu" được phân tách thành `["gilgamesh", "hates", "enkidu"]` và sau đó trở thành `[0, 24, 0]` (trong đó "hates" có chỉ số 24).

Lý tưởng nhất, chúng ta muốn mô hình có thể xử lý các token không xác định bằng cách học cách sử dụng ngữ cảnh xung quanh chúng để thực hiện dịch. Cách duy nhất để học điều đó là nếu chúng ta cũng có các token không xác định trong tập huấn luyện. Do đó, khi tạo từ điển bằng `build_vocab_from_iterator`, chúng ta sử dụng đối số `min_freq` để không tạo chỉ số cho các token xuất hiện ít hơn `min_freq` lần trong tập huấn luyện. Nói cách khác, khi sử dụng từ điển, bất kỳ token nào không xuất hiện ít nhất hai lần trong tập huấn luyện sẽ được thay thế bằng chỉ số token không xác định khi chuyển đổi token thành chỉ số.

Điều quan trọng cần lưu ý là từ điển chỉ nên được xây dựng từ tập huấn luyện và không bao giờ từ tập validation hoặc test. Điều này ngăn "rò rỉ thông tin" vào mô hình, giúp tránh việc tăng giả tạo điểm số validation/test.

Chúng ta cũng sử dụng đối số `specials` của `build_vocab_from_iterator` để truyền các _token đặc biệt_. Đây là các token mà chúng ta muốn thêm vào từ điển nhưng không nhất thiết phải xuất hiện trong các ví dụ đã được phân tách. Các token đặc biệt này sẽ xuất hiện đầu tiên trong từ điển. Chúng ta đã thảo luận về `unk_token`, `sos_token`, và `eos_token`. Token đặc biệt cuối cùng là `pad_token`, được ký hiệu là `<pad>`.

Khi đưa các câu vào mô hình, việc truyền nhiều câu cùng một lúc (được gọi là một batch) sẽ hiệu quả hơn thay vì từng câu một. Yêu cầu để các câu được gộp thành batch là tất cả chúng phải có cùng độ dài (về số lượng token). Phần lớn các câu của chúng ta không có cùng độ dài, nhưng chúng ta có thể giải quyết điều này bằng cách "đệm" (thêm token `<pad>`) phiên bản đã phân tách của mỗi câu trong một batch cho đến khi tất cả đều có số token bằng với câu dài nhất trong batch. Ví dụ, nếu chúng ta có hai câu: "I love pizza" và "I hate music videos". Chúng sẽ được phân tách thành: `["i", "love", "pizza"]` và `["i", "hate", "music", "videos"]`. Chuỗi token đầu tiên sau đó sẽ được đệm thành `["i", "love", "pizza", "<pad>"]`. Cả hai chuỗi sau đó có thể được chuyển đổi thành chỉ số sử dụng từ điển.

Đó là rất nhiều thông tin, nhưng may mắn là `torchtext` xử lý tất cả các phiền phức khi xây dựng từ điển. Chúng ta sẽ xử lý việc đệm và tạo batch sau.


In [ ]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

Bây giờ chúng ta đã có các từ điển, chúng ta có thể kiểm tra xem thực sự có gì trong đó.

Chúng ta có thể nhận mười token đầu tiên trong từ điển (chỉ số 0 đến 9) bằng phương thức `get_itos`, trong đó itos = "**i**nt **to** **s**tring" (số nguyên sang chuỗi), phương thức này trả về một danh sách các token.


In [ ]:
en_vocab.get_itos()[:10]

In [ ]:
en_vocab.get_itos()[9]

Chúng ta có thể làm tương tự cho từ điển tiếng Đức. Lưu ý rằng các token đặc biệt giống nhau và có cùng thứ tự (chỉ số 0 đến 3), tuy nhiên phần còn lại thì khác. Điều này là do các từ điển được tạo ra từ các dữ liệu khác nhau (một cái tiếng Anh và một cái tiếng Đức) mặc dù chúng đến từ cùng các ví dụ. Các chỉ số được gán cho các token không phải là token đặc biệt được sắp xếp từ xuất hiện nhiều nhất đến ít nhất (nhưng vẫn xuất hiện ít nhất `min_freq` lần).


In [ ]:
de_vocab.get_itos()[:10]

Chúng ta có thể lấy chỉ số từ một token cho trước bằng phương thức `get_stoi` (stoi = "**s**tring **to** **i**nt" - chuỗi sang số nguyên).


In [ ]:
en_vocab.get_stoi()["the"]

Để viết tắt, chúng ta có thể sử dụng từ điển như một từ điển Python thông thường và truyền token để lấy chỉ số. Lưu ý rằng điều này không hoạt động theo chiều ngược lại, tức là `en_vocab[7]` không hoạt động.


In [ ]:
en_vocab["the"]

Hàm `len` của mỗi từ điển cho chúng ta số lượng token duy nhất trong mỗi từ điển. Chúng ta có thể thấy rằng dữ liệu huấn luyện có khoảng 2000 token tiếng Đức (xuất hiện ít nhất hai lần) nhiều hơn dữ liệu tiếng Anh.


In [ ]:
len(en_vocab), len(de_vocab)

Chúng ta cũng có thể sử dụng từ khóa `in` để nhận giá trị boolean cho biết một token có trong từ điển hay không.


In [ ]:
"the" in en_vocab

Nhớ lại rằng chúng ta đã chuyển tất cả các token thành chữ thường? Điều này có nghĩa là không có token nào chứa ký tự viết hoa xuất hiện trong từ điển của chúng ta.


In [ ]:
"The" in en_vocab

Điều gì xảy ra nếu bạn cố gắng lấy chỉ số của một token không có trong từ điển? Bạn sẽ nhận được chỉ số 0 cho token `<unk>` (không xác định), đúng không?

Không hẳn. Một đặc thù của lớp từ điển `torchtext` là bạn phải thiết lập thủ công giá trị mà bạn muốn từ điển trả về khi bạn cố gắng lấy chỉ số của một token ngoài từ điển. Nếu bạn chưa thiết lập giá trị này, bạn sẽ nhận được một lỗi! Điều này là để bạn có thể thiết lập từ điển trả về bất kỳ giá trị nào khi cố gắng lấy chỉ số của một token không có trong từ điển, thậm chí là một giá trị như `-100`.


In [ ]:
# en_vocab["The"]

Chúng ta đã biết chỉ số của token `<unk>` là 0 vì nó là phần tử đầu tiên trong danh sách `special_tokens` của chúng ta, và chúng ta cũng đã kiểm tra thủ công bằng `get_itos`.

Tuy nhiên, ở đây chúng ta sẽ lấy nó theo cách lập trình và cũng kiểm tra rằng cả hai từ điển đều có cùng chỉ số cho các token không xác định và token đệm, vì điều này đơn giản hóa mã sau này.

Chúng ta cũng lấy chỉ số của token `<pad>`, vì chúng ta sẽ sử dụng nó sau này.


In [ ]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

Sử dụng phương thức `set_default_index`, chúng ta có thể thiết lập giá trị được trả về khi chúng ta cố gắng lấy chỉ số của một token ngoài từ điển. Trong trường hợp này, là chỉ số của token không xác định, `<unk>`.


In [ ]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

Bây giờ, chúng ta có thể thoải mái lấy chỉ số của các token ngoài từ điển!


In [ ]:
en_vocab["The"]

Và chúng ta có thể lấy token tương ứng với chỉ số đó để chứng minh đó là token `<unk>`.


In [ ]:
en_vocab.get_itos()[0]

Một tính năng hữu ích khác của từ điển là phương thức `lookup_indices`. Phương thức này nhận vào một danh sách các token và trả về một danh sách các chỉ số. Trong ví dụ dưới đây, chúng ta có thể thấy token "crime" không tồn tại trong từ điển nên được chuyển thành chỉ số của token `<unk>`, là 0, mà chúng ta đã truyền vào phương thức `set_default_index`.


In [ ]:
tokens = ["i", "love", "watching", "crime", "shows"]

In [ ]:
en_vocab.lookup_indices(tokens)

Ngược lại, chúng ta có thể sử dụng phương thức `lookup_tokens` để chuyển đổi một danh sách các chỉ số trở lại thành token sử dụng từ điển. Lưu ý rằng token "crime" ban đầu giờ là một token `<unk>`. Không có cách nào để biết chuỗi token ban đầu là gì.


In [ ]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

Hy vọng rằng chúng ta đã hiểu rõ cách hoạt động của lớp `torchtext.Vocab`. Đã đến lúc đưa nó vào thực hành!

Giống như `tokenize_example`, chúng ta tạo một hàm `numericalize_example` mà chúng ta sẽ sử dụng với phương thức `map` của tập dữ liệu. Hàm này sẽ "số hóa" (một cách nói hoa mỹ để chỉ việc chuyển token thành chỉ số) các token trong mỗi ví dụ sử dụng từ điển và trả về kết quả vào các đặc trưng mới "en_ids" và "de_ids".


In [ ]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

Chúng ta áp dụng hàm `numericalize_example`, truyền từ điển vào từ điển `fn_kwargs` cho đối số `fn_kwargs`.


In [ ]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Kiểm tra một ví dụ, chúng ta có thể thấy nó có hai đặc trưng mới: "en_ids" và "de_ids", cả hai đều là danh sách các số nguyên đại diện cho các chỉ số trong từ điển tương ứng.


In [ ]:
train_data[0]

Chúng ta có thể xác nhận các chỉ số là đúng bằng cách sử dụng phương thức `lookup_tokens` với từ điển tương ứng trên danh sách các chỉ số.


In [ ]:
en_vocab.lookup_tokens(train_data[0]["en_ids"])

Một điều khác mà thư viện `datasets` xử lý cho chúng ta với lớp `Dataset` là chuyển đổi các đặc trưng sang đúng kiểu. Các chỉ số trong mỗi ví dụ hiện là các số nguyên Python cơ bản. Tuy nhiên, chúng cần được chuyển đổi thành tensor PyTorch để sử dụng với PyTorch. Chúng ta có thể chuyển đổi chúng ngay trước khi truyền vào mô hình, tuy nhiên sẽ thuận tiện hơn nếu làm ngay bây giờ.

Phương thức `with_format` chuyển đổi các đặc trưng được chỉ định bởi đối số `columns` thành một `type` cho trước. Ở đây, chúng ta chỉ định kiểu là "torch" (cho PyTorch) và các cột là "en_ids" và "de_ids" (các đặc trưng mà chúng ta muốn chuyển đổi thành tensor PyTorch). Theo mặc định, `with_format` sẽ loại bỏ bất kỳ đặc trưng nào không có trong danh sách đặc trưng được truyền vào `columns`. Chúng ta muốn giữ lại các đặc trưng đó, điều có thể làm bằng `output_all_columns=True`.


In [ ]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

Chúng ta có thể xác nhận điều này đã hoạt động bằng cách kiểm tra một ví dụ và thấy các đặc trưng "en_ids" và "de_ids" được hiển thị dưới dạng `tensor([...])`.


In [ ]:
train_data[0]

Chúng ta cũng có thể kiểm tra điều này bằng hàm `type` tích hợp trên một trong các đặc trưng.


In [ ]:
type(train_data[0]["en_ids"])

## Bộ tải Dữ liệu (Data Loaders)

Bước cuối cùng của việc chuẩn bị dữ liệu là tạo các bộ tải dữ liệu. Các bộ tải này có thể được lặp qua để trả về một batch dữ liệu, mỗi batch là một từ điển chứa các câu tiếng Anh và tiếng Đức đã được số hóa (và đã được đệm) dưới dạng tensor PyTorch.

Đầu tiên, chúng ta cần tạo một hàm gom (collate), tức là kết hợp, một batch các ví dụ thành một batch. Hàm `collate_fn` dưới đây nhận vào một "batch" (một danh sách các ví dụ), sau đó chúng ta tách riêng các chỉ số tiếng Anh và tiếng Đức cho mỗi ví dụ trong batch, và truyền mỗi cái vào hàm `pad_sequence`. `pad_sequence` nhận một danh sách các tensor, đệm mỗi tensor đến độ dài của tensor dài nhất sử dụng `padding_value` (mà chúng ta đặt thành `pad_index`, chỉ số của token `<pad>`) và sau đó trả về tensor có kích thước `[độ dài tối đa, kích thước batch]`, trong đó `kích thước batch` là số lượng ví dụ trong batch và `độ dài tối đa` là độ dài của tensor dài nhất trong batch. Chúng ta đặt mỗi tensor vào một từ điển và sau đó trả về nó.

Hàm `get_collate_fn` nhận vào chỉ số token đệm và trả về hàm `collate_fn` được định nghĩa bên trong nó. Kỹ thuật này, định nghĩa một hàm bên trong một hàm khác và trả về nó, được gọi là [closure](<https://en.wikipedia.org/wiki/Closure_(computer_programming)>). Nó cho phép `collate_fn` liên tục sử dụng giá trị `pad_index` mà nó được tạo ra mà không cần tạo một lớp hoặc sử dụng biến toàn cục.


In [ ]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

Tiếp theo, chúng ta viết các hàm tạo ra các bộ tải dữ liệu sử dụng lớp `DataLoader` của PyTorch.

`get_data_loader` được tạo bằng một `Dataset`, kích thước batch, chỉ số token đệm (được sử dụng để tạo các batch trong `collate_fn`), và một giá trị boolean quyết định xem các ví dụ có nên được xáo trộn tại thời điểm bộ tải dữ liệu được lặp qua hay không.

Kích thước batch xác định số lượng ví dụ tối đa trong một batch. Nếu độ dài của tập dữ liệu không chia hết cho kích thước batch thì batch cuối cùng sẽ nhỏ hơn.


In [ ]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

Cuối cùng, chúng ta tạo các bộ tải dữ liệu.

Để giảm thời gian huấn luyện, chúng ta thường muốn sử dụng kích thước batch lớn nhất có thể. Khi sử dụng GPU, điều này có nghĩa là sử dụng kích thước batch lớn nhất vừa với bộ nhớ GPU.

Xáo trộn dữ liệu làm cho việc huấn luyện ổn định hơn và có thể cải thiện hiệu suất cuối cùng của mô hình, tuy nhiên chỉ cần thực hiện trên tập huấn luyện. Các chỉ số được tính toán cho tập validation và test sẽ giống nhau bất kể thứ tự dữ liệu.


In [ ]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

## Xây dựng Mô hình

Chúng ta sẽ xây dựng mô hình theo ba phần. Bộ mã hóa, bộ giải mã và một mô hình seq2seq bao bọc bộ mã hóa và bộ giải mã, cung cấp cách giao tiếp với từng phần.

### Bộ mã hóa (Encoder)

Đầu tiên là bộ mã hóa, một LSTM 2 lớp. Bài báo chúng ta đang triển khai sử dụng LSTM 4 lớp, nhưng để giảm thời gian huấn luyện, chúng ta cắt giảm xuống 2 lớp. Khái niệm RNN đa lớp dễ dàng mở rộng từ 2 lên 4 lớp.

Đối với RNN đa lớp, câu đầu vào, $X$, sau khi được nhúng sẽ đi vào lớp đầu tiên (dưới cùng) của RNN và các trạng thái ẩn, $H=\{h_1, h_2, ..., h_T\}$, được xuất ra bởi lớp này được sử dụng làm đầu vào cho RNN ở lớp phía trên. Do đó, biểu diễn mỗi lớp bằng một chỉ số mũ, các trạng thái ẩn ở lớp đầu tiên được cho bởi:

$$h_t^1 = \text{EncoderRNN}^1(e(x_t), h_{t-1}^1)$$

Các trạng thái ẩn ở lớp thứ hai được cho bởi:

$$h_t^2 = \text{EncoderRNN}^2(h_t^1, h_{t-1}^2)$$

Sử dụng RNN đa lớp cũng có nghĩa là chúng ta sẽ cần một trạng thái ẩn ban đầu làm đầu vào cho mỗi lớp, $h_0^l$, và chúng ta cũng sẽ xuất ra một vector ngữ cảnh cho mỗi lớp, $z^l$.

Không đi quá sâu vào chi tiết về LSTM (xem [bài viết](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) này để tìm hiểu thêm về chúng), tất cả những gì chúng ta cần biết là chúng là một loại RNN thay vì chỉ nhận vào một trạng thái ẩn và trả về một trạng thái ẩn mới mỗi bước thời gian, chúng còn nhận vào và trả về _trạng thái ô_ (cell state), $c_t$, mỗi bước thời gian.

$$
\begin{align*}
h_t &= \text{RNN}(e(x_t), h_{t-1})\\
(h_t, c_t) &= \text{LSTM}(e(x_t), h_{t-1}, c_{t-1})
\end{align*}
$$

Chúng ta có thể đơn giản coi $c_t$ là một loại trạng thái ẩn khác. Tương tự như $h_0^l$, $c_0^l$ sẽ được khởi tạo thành một tensor toàn số 0. Ngoài ra, vector ngữ cảnh của chúng ta bây giờ sẽ bao gồm cả trạng thái ẩn cuối cùng và trạng thái ô cuối cùng, tức là $z^l = (h_T^l, c_T^l)$.

Mở rộng các phương trình đa lớp cho LSTM, chúng ta có:

$$
\begin{align*}
(h_t^1, c_t^1) &= \text{EncoderLSTM}^1(e(x_t), (h_{t-1}^1, c_{t-1}^1))\\
(h_t^2, c_t^2) &= \text{EncoderLSTM}^2(h_t^1, (h_{t-1}^2, c_{t-1}^2))
\end{align*}
$$

Lưu ý rằng chỉ trạng thái ẩn từ lớp đầu tiên được truyền làm đầu vào cho lớp thứ hai, chứ không phải trạng thái ô.

Vì vậy, bộ mã hóa của chúng ta trông giống như thế này:

![](assets/seq2seq2.png)

Chúng ta tạo điều này trong mã bằng cách tạo một module `Encoder`, yêu cầu chúng ta kế thừa từ `torch.nn.Module` và sử dụng `super().__init__()` làm mã mẫu. Bộ mã hóa nhận các đối số sau:

-   `input_dim` là kích thước/chiều của vector one-hot sẽ được đưa vào bộ mã hóa. Giá trị này bằng với kích thước từ điển đầu vào (nguồn).
-   `embedding_dim` là chiều của lớp nhúng. Lớp này chuyển các vector one-hot thành vector dày đặc với `embedding_dim` chiều.
-   `hidden_dim` là chiều của trạng thái ẩn và trạng thái ô.
-   `n_layers` là số lượng lớp trong RNN.
-   `dropout` là lượng dropout để sử dụng. Đây là một tham số điều chuẩn để ngăn quá khớp (overfitting). Xem [link](https://www.coursera.org/lecture/deep-neural-network/understanding-dropout-YaGbR) này để biết thêm chi tiết về dropout.

Chúng ta sẽ không thảo luận chi tiết về lớp nhúng trong các hướng dẫn này. Tất cả những gì chúng ta cần biết là có một bước trước khi các từ - về mặt kỹ thuật, các chỉ số của các từ - được truyền vào RNN, nơi các từ được chuyển đổi thành vector. Để đọc thêm về nhúng từ, xem các bài viết: [1](https://monkeylearn.com/blog/word-embeddings-transform-text-numbers/), [2](http://p.migdal.pl/2017/01/06/king-man-woman-queen-why.html), [3](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/), [4](http://mccormickml.com/2017/01/11/word2vec-tutorial-part-2-negative-sampling/).

Lớp nhúng được tạo bằng `nn.Embedding`, LSTM bằng `nn.LSTM` và lớp dropout bằng `nn.Dropout`. Xem [tài liệu](https://pytorch.org/docs/stable/nn.html) PyTorch để biết thêm về các lớp này.

Một điều cần lưu ý là đối số `dropout` của LSTM là lượng dropout được áp dụng giữa các lớp của RNN đa lớp, tức là giữa các trạng thái ẩn được xuất ra từ lớp $l$ và các trạng thái ẩn đó được sử dụng làm đầu vào cho lớp $l+1$.

Trong phương thức `forward`, chúng ta truyền vào câu nguồn, $X$, câu này được chuyển đổi thành vector dày đặc sử dụng lớp `embedding`, và sau đó dropout được áp dụng. Các nhúng này sau đó được truyền vào RNN. Vì chúng ta truyền toàn bộ chuỗi vào RNN, nó sẽ tự động thực hiện tính toán hồi quy các trạng thái ẩn trên toàn bộ chuỗi cho chúng ta! Lưu ý rằng chúng ta không truyền trạng thái ẩn hoặc trạng thái ô ban đầu vào RNN. Điều này là vì, như đã ghi trong [tài liệu](https://pytorch.org/docs/stable/nn.html#torch.nn.LSTM), nếu không truyền trạng thái ẩn/ô vào RNN, nó sẽ tự động tạo một trạng thái ẩn/ô ban đầu dưới dạng tensor toàn số 0.

RNN trả về: `outputs` (trạng thái ẩn của lớp trên cùng cho mỗi bước thời gian), `hidden` (trạng thái ẩn cuối cùng cho mỗi lớp, $h_T$, được xếp chồng lên nhau) và `cell` (trạng thái ô cuối cùng cho mỗi lớp, $c_T$, được xếp chồng lên nhau).

Vì chúng ta chỉ cần các trạng thái ẩn và ô cuối cùng (để tạo vector ngữ cảnh), `forward` chỉ trả về `hidden` và `cell`.

Kích thước của mỗi tensor được để lại dưới dạng bình luận trong mã. Trong triển khai này, `n_directions` sẽ luôn là 1, tuy nhiên lưu ý rằng RNN hai chiều (được đề cập trong hướng dẫn 3) sẽ có `n_directions` là 2.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

### Bộ giải mã (Decoder)

Tiếp theo, chúng ta sẽ xây dựng bộ giải mã, bộ này cũng sẽ là một LSTM 2 lớp (4 lớp trong bài báo).

![](assets/seq2seq3.png)

Lớp `Decoder` thực hiện một bước giải mã duy nhất, tức là xuất ra một token duy nhất mỗi bước thời gian. Lớp đầu tiên sẽ nhận trạng thái ẩn và trạng thái ô từ bước thời gian trước đó, $(s_{t-1}^1, c_{t-1}^1)$, và đưa nó qua LSTM cùng với token đã nhúng hiện tại, $y_t$, để tạo ra trạng thái ẩn và trạng thái ô mới, $(s_t^1, c_t^1)$. Các lớp tiếp theo sẽ sử dụng trạng thái ẩn từ lớp phía dưới, $s_t^{l-1}$, và các trạng thái ẩn và ô trước đó từ lớp của chúng, $(s_{t-1}^l, c_{t-1}^l)$. Điều này cung cấp các phương trình rất giống với các phương trình trong bộ mã hóa.

$$
\begin{align*}
(s_t^1, c_t^1) = \text{DecoderLSTM}^1(d(y_t), (s_{t-1}^1, c_{t-1}^1))\\
(s_t^2, c_t^2) = \text{DecoderLSTM}^2(s_t^1, (s_{t-1}^2, c_{t-1}^2))
\end{align*}
$$

Nhớ rằng các trạng thái ẩn và ô ban đầu của bộ giải mã là các vector ngữ cảnh, là các trạng thái ẩn và ô cuối cùng của bộ mã hóa từ cùng một lớp, tức là $(s_0^l,c_0^l)=z^l=(h_T^l,c_T^l)$.

Sau đó, chúng ta truyền trạng thái ẩn từ lớp trên cùng của RNN, $s_t^L$, qua một lớp tuyến tính, $f$, để đưa ra dự đoán token tiếp theo trong chuỗi đích (đầu ra), $\hat{y}_{t+1}$.

$$\hat{y}_{t+1} = f(s_t^L)$$

Các đối số và khởi tạo tương tự như lớp `Encoder`, ngoại trừ việc bây giờ chúng ta có `output_dim` là kích thước từ điển của ngôn ngữ đầu ra/đích. Cũng có thêm lớp `Linear`, được sử dụng để đưa ra các dự đoán từ trạng thái ẩn của lớp trên cùng.

Trong phương thức `forward`, chúng ta nhận một batch các token đầu vào, các trạng thái ẩn trước đó và các trạng thái ô trước đó. Vì chúng ta chỉ giải mã một token tại một thời điểm, các token đầu vào sẽ luôn có độ dài chuỗi là 1. Chúng ta `unsqueeze` các token đầu vào để thêm một chiều độ dài câu bằng 1. Sau đó, tương tự như bộ mã hóa, chúng ta truyền qua lớp nhúng và áp dụng dropout. Batch các token đã nhúng này sau đó được truyền vào RNN cùng với các trạng thái ẩn và ô trước đó. Điều này tạo ra một `output` (trạng thái ẩn từ lớp trên cùng của RNN), một trạng thái `hidden` mới (mỗi lớp một cái, xếp chồng lên nhau) và một trạng thái `cell` mới (cũng mỗi lớp một cái, xếp chồng lên nhau). Sau đó, chúng ta truyền `output` (sau khi loại bỏ chiều độ dài câu) qua lớp tuyến tính để nhận `prediction`. Sau đó, chúng ta trả về `prediction`, trạng thái `hidden` mới và trạng thái `cell` mới.

**Lưu ý**: vì chúng ta luôn có độ dài chuỗi là 1, chúng ta có thể sử dụng `nn.LSTMCell` thay vì `nn.LSTM`, vì nó được thiết kế để xử lý một batch đầu vào không nhất thiết phải là một chuỗi. `nn.LSTMCell` chỉ là một ô đơn lẻ và `nn.LSTM` là một wrapper xung quanh nhiều ô tiềm năng. Sử dụng `nn.LSTMCell` trong trường hợp này có nghĩa là chúng ta không cần `unsqueeze` để thêm một chiều độ dài chuỗi giả, nhưng chúng ta sẽ cần một `nn.LSTMCell` cho mỗi lớp trong bộ giải mã và phải đảm bảo mỗi `nn.LSTMCell` nhận đúng trạng thái ẩn ban đầu từ bộ mã hóa. Tất cả những điều này làm cho mã kém súc tích hơn -- do đó quyết định tiếp tục sử dụng `nn.LSTM` thông thường.


In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

### Seq2Seq

Đối với phần cuối cùng của triển khai, chúng ta sẽ triển khai mô hình seq2seq. Mô hình này sẽ xử lý:

-   nhận câu đầu vào/nguồn
-   sử dụng bộ mã hóa để tạo ra các vector ngữ cảnh
-   sử dụng bộ giải mã để tạo ra câu đầu ra/đích được dự đoán

Mô hình đầy đủ của chúng ta sẽ trông như thế này:

![](assets/seq2seq4.png)

Mô hình `Seq2Seq` nhận vào một `Encoder`, `Decoder`, và một `device` (được sử dụng để đặt tensor trên GPU, nếu có).

Đối với triển khai này, chúng ta phải đảm bảo rằng số lượng lớp và chiều của trạng thái ẩn (và ô) bằng nhau trong `Encoder` và `Decoder`. Điều này không phải lúc nào cũng đúng, chúng ta không nhất thiết cần cùng số lượng lớp hoặc cùng kích thước chiều ẩn trong một mô hình sequence-to-sequence. Tuy nhiên, nếu chúng ta làm điều gì đó như có số lượng lớp khác nhau thì chúng ta sẽ cần đưa ra quyết định về cách xử lý. Ví dụ, nếu bộ mã hóa có 2 lớp và bộ giải mã chỉ có 1 lớp, làm thế nào để xử lý? Chúng ta có tính trung bình hai vector ngữ cảnh được xuất ra bởi bộ giải mã? Truyền cả hai qua một lớp tuyến tính? Chỉ sử dụng vector ngữ cảnh từ lớp cao nhất? V.v.

Phương thức `forward` của chúng ta nhận câu nguồn, câu đích và tỷ lệ teacher forcing. Tỷ lệ teacher forcing được sử dụng khi huấn luyện mô hình. Khi giải mã, tại mỗi bước thời gian chúng ta sẽ dự đoán token tiếp theo trong chuỗi đích từ các token đã giải mã trước đó, $\hat{y}_{t+1}=f(s_t^L)$. Với xác suất bằng tỷ lệ teacher forcing (`teacher_forcing_ratio`), chúng ta sẽ sử dụng token tiếp theo thực tế (ground truth) trong chuỗi làm đầu vào cho bộ giải mã trong bước thời gian tiếp theo. Tuy nhiên, với xác suất `1 - teacher_forcing_ratio`, chúng ta sẽ sử dụng token mà mô hình dự đoán làm đầu vào tiếp theo cho mô hình, ngay cả khi nó không khớp với token tiếp theo thực tế trong chuỗi.

Điều đầu tiên chúng ta làm trong phương thức `forward` là tạo một tensor `outputs` sẽ lưu trữ tất cả các dự đoán của chúng ta, $\hat{Y}$.

Sau đó, chúng ta đưa câu đầu vào/nguồn, `src`, vào bộ mã hóa và nhận các trạng thái ẩn và ô cuối cùng.

Đầu vào đầu tiên cho bộ giải mã là token bắt đầu chuỗi (`<sos>`). Vì tensor `trg` của chúng ta đã có token `<sos>` được thêm vào (ngay từ khi chúng ta phân tách các câu tiếng Anh), chúng ta lấy $y_1$ bằng cách cắt vào nó. Chúng ta biết câu đích của chúng ta nên dài bao nhiêu (`trg_length`), vì vậy chúng ta lặp lại nhiều lần như vậy. Token cuối cùng được đưa vào bộ giải mã là token **trước** token `<eos>` -- token `<eos>` không bao giờ được đưa vào bộ giải mã.

Trong mỗi lần lặp, chúng ta:

-   truyền đầu vào, các trạng thái ẩn trước đó và trạng thái ô trước đó ($y_t, s_{t-1}, c_{t-1}$) vào bộ giải mã
-   nhận một dự đoán, trạng thái ẩn tiếp theo và trạng thái ô tiếp theo ($\hat{y}_{t+1}, s_{t}, c_{t}$) từ bộ giải mã
-   đặt dự đoán của chúng ta, $\hat{y}_{t+1}$/`output` vào tensor dự đoán, $\hat{Y}$/`outputs`
-   quyết định xem chúng ta có sử dụng "teacher forcing" hay không
    -   nếu có, `input` tiếp theo là token tiếp theo thực tế trong chuỗi, $y_{t+1}$/`trg[t]`
    -   nếu không, `input` tiếp theo là token tiếp theo được dự đoán trong chuỗi, $\hat{y}_{t+1}$/`top1`, mà chúng ta lấy bằng cách thực hiện `argmax` trên tensor đầu ra

Sau khi đã đưa ra tất cả các dự đoán, chúng ta trả về tensor đầy các dự đoán, $\hat{Y}$/`outputs`.

**Lưu ý**: vòng lặp giải mã của chúng ta bắt đầu từ 1, không phải 0. Điều này có nghĩa là phần tử thứ 0 của tensor `outputs` vẫn toàn số 0. Vì vậy, `trg` và `outputs` của chúng ta trông giống như:

$$
\begin{align*}
\text{trg} = [<sos>, &y_1, y_2, y_3, <eos>]\\
\text{outputs} = [0, &\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

Sau này khi tính toán mất mát, chúng ta cắt bỏ phần tử đầu tiên của mỗi tensor để được:

$$
\begin{align*}
\text{trg} = [&y_1, y_2, y_3, <eos>]\\
\text{outputs} = [&\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

## Huấn luyện Mô hình

Bây giờ chúng ta đã triển khai xong mô hình, chúng ta có thể bắt đầu huấn luyện nó.

### Khởi tạo Mô hình

Đầu tiên, chúng ta sẽ khởi tạo mô hình. Như đã đề cập trước đó, các chiều đầu vào và đầu ra được xác định bởi kích thước từ điển. Chiều nhúng và dropout cho bộ mã hóa và bộ giải mã có thể khác nhau, nhưng số lượng lớp và kích thước của trạng thái ẩn/ô phải giống nhau.

Sau đó, chúng ta định nghĩa bộ mã hóa, bộ giải mã và mô hình Seq2Seq, mà chúng ta đặt trên `device`. `device` được sử dụng để nói cho PyTorch biết liệu một mô hình hoặc tensor nên được xử lý trên GPU hay CPU. Hàm `torch.cuda.is_available()` trả về `True` nếu phát hiện GPU trên máy của chúng ta. Do đó, mô hình của chúng ta sẽ được đặt trên GPU, nếu chúng ta có một cái.


In [ ]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

### Khởi tạo Trọng số

Tiếp theo là khởi tạo trọng số của mô hình. Trong bài báo, họ cho biết họ khởi tạo tất cả trọng số từ phân phối đều giữa -0.08 và +0.08, tức là $\mathcal{U}(-0.08, 0.08)$.

Chúng ta khởi tạo trọng số trong PyTorch bằng cách tạo một hàm mà chúng ta `apply` vào mô hình. Khi sử dụng `apply`, hàm `init_weights` sẽ được gọi trên mỗi module và sub-module trong mô hình. Đối với mỗi module, chúng ta lặp qua tất cả các tham số và lấy mẫu từ phân phối đều bằng `nn.init.uniform_`.


In [ ]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Chúng ta cũng có thể đếm số lượng tham số trong mô hình.


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

### Bộ tối ưu hóa (Optimizer)

Chúng ta định nghĩa bộ tối ưu hóa, được sử dụng để cập nhật các tham số trong vòng lặp huấn luyện. Xem [bài viết](http://ruder.io/optimizing-gradient-descent/) này để biết thông tin về các bộ tối ưu hóa khác nhau. Ở đây, chúng ta sẽ sử dụng Adam.


In [ ]:
optimizer = optim.Adam(model.parameters())

### Hàm Mất mát

Tiếp theo, chúng ta định nghĩa hàm mất mát. Hàm `CrossEntropyLoss` tính toán cả log softmax cũng như log-likelihood âm của các dự đoán.

Hàm mất mát của chúng ta tính toán mất mát trung bình cho mỗi token, tuy nhiên bằng cách truyền chỉ số của token `<pad>` làm đối số `ignore_index`, chúng ta bỏ qua mất mát bất cứ khi nào token đích là token đệm.


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

### Vòng lặp Huấn luyện

Tiếp theo, chúng ta sẽ định nghĩa vòng lặp huấn luyện.

Đầu tiên, chúng ta sẽ đặt mô hình vào "chế độ huấn luyện" bằng `model.train()`. Điều này sẽ bật dropout (và chuẩn hóa batch, mà chúng ta không sử dụng) và sau đó lặp qua bộ lặp dữ liệu.

Như đã nói trước đó, vòng lặp giải mã bắt đầu từ 1, không phải 0. Điều này có nghĩa là phần tử thứ 0 của tensor `outputs` vẫn toàn số 0. Vì vậy, `trg` và `outputs` trông giống như:

$$
\begin{align*}
\text{trg} = [<sos>, &y_1, y_2, y_3, <eos>]\\
\text{outputs} = [0, &\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

Ở đây, khi tính toán mất mát, chúng ta cắt bỏ phần tử đầu tiên của mỗi tensor để được:

$$
\begin{align*}
\text{trg} = [&y_1, y_2, y_3, <eos>]\\
\text{outputs} = [&\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

Tại mỗi lần lặp:

-   lấy câu nguồn và câu đích từ batch, $X$ và $Y$
-   đặt lại các gradient được tính toán từ batch trước đó về 0
-   đưa nguồn và đích vào mô hình để nhận đầu ra, $\hat{Y}$
-   vì hàm mất mát chỉ hoạt động trên đầu vào 2d với đích 1d, chúng ta cần làm phẳng mỗi cái bằng `.view`
    -   chúng ta cắt bỏ cột đầu tiên của tensor đầu ra và đích như đã đề cập ở trên
-   tính toán các gradient bằng `loss.backward()`
-   cắt gradient để ngăn chúng bùng nổ (một vấn đề phổ biến trong RNN)
-   cập nhật các tham số của mô hình bằng cách thực hiện một bước tối ưu hóa
-   cộng dồn giá trị mất mát vào tổng đang chạy

Cuối cùng, chúng ta trả về mất mát được trung bình trên tất cả các batch.


In [ ]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

### Vòng lặp Đánh giá

Vòng lặp đánh giá của chúng ta tương tự như vòng lặp huấn luyện, tuy nhiên vì chúng ta không cập nhật bất kỳ tham số nào nên không cần truyền bộ tối ưu hóa hay giá trị cắt gradient.

Chúng ta phải nhớ đặt mô hình ở chế độ đánh giá bằng `model.eval()`. Điều này sẽ tắt dropout (và chuẩn hóa batch, nếu sử dụng).

Chúng ta sử dụng khối `with torch.no_grad()` để đảm bảo không có gradient nào được tính toán trong khối. Điều này giảm tiêu thụ bộ nhớ và tăng tốc độ.

Vòng lặp lặp tương tự (không có cập nhật tham số), tuy nhiên chúng ta phải đảm bảo tắt teacher forcing khi đánh giá. Điều này sẽ khiến mô hình chỉ sử dụng các dự đoán của chính nó để đưa ra các dự đoán tiếp theo trong một câu, điều này phản ánh cách nó sẽ được sử dụng khi triển khai thực tế.


In [ ]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

### Huấn luyện Mô hình

Cuối cùng chúng ta có thể bắt đầu huấn luyện mô hình!

Tại mỗi epoch, chúng ta sẽ kiểm tra xem mô hình đã đạt được mất mát validation tốt nhất cho đến nay chưa. Nếu rồi, chúng ta sẽ cập nhật mất mát validation tốt nhất và lưu các tham số của mô hình (được gọi là `state_dict` trong PyTorch). Sau đó, khi kiểm tra mô hình, chúng ta sẽ sử dụng các tham số đã lưu để đạt được mất mát validation tốt nhất.

Chúng ta sẽ in ra cả mất mát và độ bối rối (perplexity) tại mỗi epoch. Dễ thấy sự thay đổi về perplexity hơn là sự thay đổi về mất mát vì các số lớn hơn nhiều.


In [ ]:
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

Chúng ta đã huấn luyện thành công một mô hình dịch tiếng Đức sang tiếng Anh! Nhưng mô hình hoạt động tốt đến đâu?


## Đánh giá Mô hình

Điều đầu tiên cần làm là kiểm tra hiệu suất của mô hình trên tập test.

Chúng ta sẽ tải các tham số (`state_dict`) giúp mô hình đạt mất mát validation tốt nhất và chạy nó trên tập test để lấy mất mát và perplexity trên tập test.


In [ ]:
model.load_state_dict(torch.load("tut1-model.pt"))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

Khá tương tự với hiệu suất trên tập validation, đây là một dấu hiệu tốt. Điều này có nghĩa là chúng ta không bị quá khớp trên tập validation.

Bạn có thể nghĩ rằng không thể quá khớp trên tập validation, nhưng không phải vậy. Mỗi khi bạn điều chỉnh siêu tham số (ví dụ: bộ tối ưu hóa, tốc độ học, kiến trúc mô hình, khởi tạo trọng số, v.v.) để có kết quả tốt hơn trên tập validation, bạn đang dần quá khớp các siêu tham số đó vào tập validation. Bạn cũng có thể làm điều này trên tập test! Do đó, bạn nên đánh giá mô hình trên tập test càng ít lần càng tốt.

Hầu hết các bài báo sử dụng mạng nơ-ron cho dịch máy không đưa ra kết quả dưới dạng mất mát và perplexity trên tập test, họ thường đưa ra điểm [BLEU](https://en.wikipedia.org/wiki/BLEU). Khác với mất mát/perplexity, BLEU là một giá trị giữa 0 và 1, trong đó giá trị cao hơn là tốt hơn, và [theo bài báo BLEU gốc](https://aclanthology.org/P02-1040.pdf), nó có tương quan cao với đánh giá của con người.

Để lấy điểm BLEU của mô hình trên tập test, trước tiên chúng ta cần sử dụng mô hình để dịch mọi ví dụ từ tập test, điều mà chúng ta thực hiện bằng hàm `translate_sentence` dưới đây. Hàm này đầu tiên chuyển `sentence` đầu vào thành các `token`, tùy chọn chuyển mỗi `token` thành chữ thường, và sau đó thêm các token bắt đầu và kết thúc chuỗi, `sos_token` và `eos_token`. Sau đó, nó sử dụng từ điển để số hóa các token thành `ids` và chuyển các `ids` này thành một `tensor`, thêm một chiều batch "giả", và sau đó truyền `tensor` qua `encoder` để lấy các trạng thái `hidden` và `cell`. Sau đó, chúng ta thực hiện giải mã, bắt đầu với `sos_token`, chuyển nó thành tensor, truyền qua `decoder`, lấy `predicted_token` mà mô hình cho là có khả năng nhất là token tiếp theo trong chuỗi, mà chúng ta thêm vào danh sách các `inputs` cho bộ giải mã. Nếu `predicted_token` là token kết thúc chuỗi thì chúng ta dừng giải mã, nếu không chúng ta tiếp tục vòng lặp, sử dụng `predicted_token` làm đầu vào tiếp theo cho bộ giải mã. Chúng ta tiếp tục giải mã cho đến khi bộ giải mã xuất ra `eos_token` hoặc đạt đến `max_output_length` (mà chúng ta sử dụng để tránh việc bộ giải mã cứ tạo token mãi mãi). Sau khi đã dừng giải mã, chúng ta chuyển các đầu vào thành `tokens` sử dụng từ điển và trả về chúng.


In [ ]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

Chúng ta sẽ truyền một ví dụ test (một thứ mà mô hình chưa được huấn luyện) để sử dụng làm câu kiểm tra hàm `translate_sentence`, truyền vào câu tiếng Đức và mong nhận được một thứ gì đó trông giống câu tiếng Anh.


In [ ]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

In [ ]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

Mô hình dường như đã nhận ra rằng câu đầu vào đề cập đến một người đàn ông đang mặc một món đồ nào đó (mặc dù cả màu sắc và món đồ đều sai), nhưng nó dường như không thể xác định được người đàn ông đang làm gì.

Chúng ta không nên kỳ vọng kết quả quá tuyệt vời, mô hình của chúng ta khá nhỏ so với những gì được sử dụng trong bài báo chúng ta đang triển khai (họ sử dụng bốn lớp với chiều nhúng và chiều ẩn là 1000) và rất nhỏ so với các mô hình dịch hiện đại (có hàng tỷ tham số).


In [ ]:
translation

Mô hình không chỉ dịch các ví dụ trong tập huấn luyện, validation và test. Chúng ta có thể sử dụng nó để dịch các câu tùy ý bằng cách truyền bất kỳ chuỗi nào vào `translate_sentence`.

Lưu ý rằng tập dữ liệu multi30k bao gồm các chú thích hình ảnh đã được dịch từ tiếng Anh sang tiếng Đức, và mô hình của chúng ta đã được huấn luyện để dịch từ tiếng Đức sang tiếng Anh. Do đó, mô hình sẽ chỉ đưa ra các bản dịch hợp lý nếu các câu là câu tiếng Đức có thể là chú thích hình ảnh. (Cũng cần nhấn mạnh lại rằng mô hình được huấn luyện ở đây khá nhỏ và hiệu suất dịch nhìn chung sẽ kém.)

Dưới đây, chúng ta nhập bản dịch tiếng Đức của câu **"Ein Mann sitzt auf einer Bank."**


In [ ]:
sentence = "Ein Mann sitzt auf einer Bank."

In [ ]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

Và chúng ta nhận được bản dịch, khá gần với bản gốc.


In [ ]:
translation

Bây giờ chúng ta có thể lặp qua `test_data`, lấy bản dịch của mô hình cho mỗi câu test.


In [ ]:
translations = [
    translate_sentence(
        example["de"],
        model,
        en_nlp,
        de_nlp,
        en_vocab,
        de_vocab,
        lower,
        sos_token,
        eos_token,
        device,
    )
    for example in tqdm.tqdm(test_data)
]

Để tính BLEU, chúng ta sẽ sử dụng thư viện `evaluate`. Khuyến nghị sử dụng các thư viện để đo lường các chỉ số để đảm bảo không có lỗi trong tính toán chỉ số, tránh đưa ra kết quả có thể không chính xác.

Chỉ số BLEU có thể được tải từ thư viện `evaluate` như sau:


In [ ]:
bleu = evaluate.load("bleu")

Một đặc thù của chỉ số BLEU là nó mong đợi các dự đoán (bản dịch được dự đoán) là chuỗi và các tham chiếu (câu tiếng Anh thực tế) là một danh sách các câu. Điều này là do BLEU hoạt động nếu bạn có nhiều câu đúng cho mỗi dự đoán vì có thể có nhiều cách dịch một câu. Trong trường hợp của chúng ta, chúng ta chỉ có một câu tham chiếu duy nhất nên chúng ta chỉ bọc câu đích trong một danh sách. Chúng ta cũng chuyển các bản dịch từ danh sách token thành chuỗi bằng cách nối chúng với khoảng trắng ở giữa và loại bỏ các token `<sos>` và `<eos>` (vì chúng sẽ không bao giờ xuất hiện trong câu tham chiếu).


In [ ]:
predictions = [" ".join(translation[1:-1]) for translation in translations]

references = [[example["en"]] for example in test_data]

In [ ]:
predictions[0], references[0]

Chúng ta cũng cần định nghĩa một hàm phân tách từ cho một chuỗi đầu vào. Hàm này sẽ được sử dụng để tính điểm BLEU bằng cách so sánh các token được dự đoán với các token tham chiếu.

Có vẻ hơi kỳ lạ là chúng ta nối các token đã dịch lại thành chuỗi chỉ để lại phân tách chúng, và cũng sử dụng chuỗi tiếng Anh từ dữ liệu test thay vì các token hiện có (`en_tokens`), tuy nhiên đây là một đặc thù khác của chỉ số BLEU được cung cấp bởi thư viện `evaluate`; `predictions` và `references` phải là chuỗi chứ không phải token, và chúng ta phải nói cho chỉ số cách các chuỗi này nên được phân tách.

Hàm `get_tokenize_fn` trả về `tokenizer_fn`, hàm này sử dụng bộ phân tách từ `spaCy` và chuyển token thành chữ thường nếu cần.


In [ ]:
def get_tokenizer_fn(nlp, lower):
    def tokenizer_fn(s):
        tokens = [token.text for token in nlp.tokenizer(s)]
        if lower:
            tokens = [token.lower() for token in tokens]
        return tokens

    return tokenizer_fn

In [ ]:
tokenizer_fn = get_tokenizer_fn(en_nlp, lower)

In [ ]:
tokenizer_fn(predictions[0]), tokenizer_fn(references[0][0])

Cuối cùng, chúng ta tính chỉ số BLEU trên toàn bộ tập test!

Chúng ta truyền `predictions`, `references` và `tokenizer_fn` vào phương thức `compute` của chỉ số BLEU để lấy kết quả.


In [ ]:
results = bleu.compute(
    predictions=predictions, references=references, tokenizer=tokenizer_fn
)

In [ ]:
results

Chúng ta nhận được điểm BLEU là 0.14! Không có gì để khoe khoan, nhưng không tệ cho mô hình dịch đầu tiên của chúng ta.

Trong các notebook tiếp theo, chúng ta sẽ triển khai thêm các bài báo về dịch máy và từng bước nâng cao điểm BLEU đạt được.
